# Notebook 6: Explaining Denied Claims

This notebook brings everything together: SHACL violations identify *why* claims were denied, and Bedrock generates customer-facing explanations grounded in graph evidence.


---
### Setup


In [ ]:
%graph_notebook_config --store-to NOTEBOOK_CONFIG --silent

In [ ]:
from workshop import *
import json
configure(neptune_config=json.loads(NOTEBOOK_CONFIG))

# Load ontology + data into a local graph (needed for SHACL validation)
g = load_ontology()
data_graph, data_source = load_claim_data()
g += data_graph
print(f"Graph: {len(g)} triples ({data_source})")

---
## Module 8: Deep Dive - Why Was Claim C003 Denied?

Claim C003 is the key example. The claim was denied, but there's no denial code in the data.  
Instead, the SHACL eligibility rule `ClaimHasActiveCoverageShape` detected that Policy P002 had a **coverage gap** during the incident date.

Let's trace through this step by step.

In [ ]:
# Step 1: Get the full context for claim C003
CLAIM_URI = "http://example.org/insurance#claim003"

context_query = """
PREFIX : <http://example.org/insurance#>
SELECT ?claimId ?claimAmount ?incidentDate ?policyId
       ?eventType ?eventTimestamp ?eventAmount
       ?coverageAmount ?deductible ?validFrom ?validTo ?isCurrent
WHERE {
    BIND(<http://example.org/insurance#claim003> AS ?claim)
    ?claim :claimId ?claimId ;
           :claimAmount ?claimAmount ;
           :incidentDate ?incidentDate ;
           :forPolicy ?policy .
    ?policy :policyId ?policyId .
    ?claim :hasEvent ?event .
    ?event :eventType ?eventType ;
           :eventTimestamp ?eventTimestamp .
    OPTIONAL { ?event :eventAmount ?eventAmount }
    ?policy :hasCoverage ?coverage .
    ?coverage :coverageAmount ?coverageAmount ;
             :deductible ?deductible ;
             :validFrom ?validFrom ;
             :isCurrent ?isCurrent .
    OPTIONAL { ?coverage :validTo ?validTo }
}
ORDER BY ?eventTimestamp ?validFrom
"""

rows = sparql_query(context_query)

# Display claim info
r = rows[0]
print(f"Claim: {r['claimId']}")
print(f"Amount: ${float(r['claimAmount']):,.2f}")
print(f"Incident Date: {r['incidentDate'][:10]}")
print(f"Policy: {r['policyId']}")

# Display events
print(f"\nEvent Timeline:")
seen_events = set()
for r in rows:
    evt_key = r['eventTimestamp']
    if evt_key not in seen_events:
        seen_events.add(evt_key)
        amt = f"  ${float(r['eventAmount']):,.2f}" if r.get('eventAmount') else ""
        print(f"  {r['eventTimestamp'][:10]}  {r['eventType']:<12}{amt}")

# Display coverage history
print(f"\nPolicy Coverage History (SCD2):")
seen_cov = set()
for r in rows:
    cov_key = r['validFrom']
    if cov_key not in seen_cov:
        seen_cov.add(cov_key)
        valid_to = r['validTo'][:10] if r.get('validTo') else 'present'
        current = 'CURRENT' if r['isCurrent'] in ('true', True) else 'EXPIRED'
        print(f"  ${float(r['coverageAmount']):>10,.2f}  {r['validFrom'][:10]} → {valid_to}  ({current})")

### Step 2: Visualize the Coverage Gap

The timeline below shows exactly why Claim C003 was denied. Policy P002's coverage v1 expired on 2025-12-31, and coverage v2 didn't start until 2026-02-01. The incident on 2026-01-15 falls squarely in the gap - this is what the `ClaimHasActiveCoverageShape` SHACL rule detected:

```
Policy P002 Coverage Timeline:

2025-06    2025-12-31    2026-01-15    2026-02-01
  │           │              │              │
  ├───────────┤              │              ├──────────── → present
  │  v1: $75K │              │              │  v2: $90K
  │  EXPIRED  │   ◄──────── GAP ────────►   │  CURRENT
  └───────────┘              │              └────────────
                             │
                     C003 incident
                     ($3,000 claim)

  Coverage v1 expired 2025-12-31
  Coverage v2 started 2026-02-01
  Incident occurred 2026-01-15 — DURING THE GAP

  → ClaimHasActiveCoverageShape fires: no active coverage at incident date
```

---
## Module 9: Generating the Denial Explanation

Now we feed the claim context + SHACL violations to Bedrock and ask it to generate a **customer-facing explanation**.

This is the same pipeline used in production:
1. SHACL violations provide the *machine-readable* denial reason
2. Claim context provides the *supporting data* (dates, amounts, coverage history)
3. Bedrock LLM translates both into a *human-readable* explanation

In [ ]:
# ── Module 9 Setup: Build denied_claims, all_violations, data_graph ───
from explain_denial import find_denied_claims, get_claim_context, validate_claim_eligibility, load_eligibility_shapes, DenialExplainer

data_graph = g

denied_claims = find_denied_claims(data_graph)
print(f"Denied claims found: {len(denied_claims)}")
for dc in denied_claims:
    print(f"  {dc['claim_id']}  ${dc['claim_amount']:,.2f}  Policy: {dc['policy_id']}  Incident: {dc['incident_date'][:10]}")

elig_shapes_graph = load_eligibility_shapes()
_, all_violations, _ = validate_claim_eligibility(data_graph, elig_shapes_graph)
print(f"\nClaims with eligibility violations: {len(all_violations)}")
for uri, viols in all_violations.items():
    claim_id = uri.split('#')[-1]
    print(f"  {claim_id}: {len(viols)} violation(s)")


### Generate Denial Explanations

Now we bring everything together. For each denied claim, we send the SHACL violation report and the full claim context (events, coverage history) to Bedrock. The LLM generates a customer-facing explanation that is **grounded in the graph evidence** - it explains what the rules determined, not what it imagines:


In [ ]:
import boto3
bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")
import textwrap
# ── Explain each denied claim ─────────────────────────────────────────
# Uses Bedrock Haiku by default (fast, cost-effective)
# Change model to "sonnet" for highest quality, or "nova-lite" for cheapest

def explain_with_fallback(denied_claims, all_violations, data_graph):
    """Try Bedrock explanation; fall back to raw SHACL violations."""
    try:
        explainer = DenialExplainer(model="haiku", region="us-east-1")
        print(f"Model: {explainer.model_id}\n")
        print("=" * 70)
            
        for claim in denied_claims:
            claim_uri = claim["claim_uri"]
            claim_viols = all_violations.get(claim_uri, [])
            claim_ctx = get_claim_context(data_graph, claim_uri)
                
            print(f"\nClaim {claim['claim_id']} (${claim['claim_amount']:,.2f})")
            print(f"Policy: {claim['policy_id']} | Incident: {claim['incident_date'][:10]}")
            print(f"Violations: {len(claim_viols)}")
            for v in claim_viols:
                print(f"  → {v['message'][:100]}")
                
            if claim_viols:
                print(f"\nGenerating explanation...")
                result = explainer.explain(claim_ctx, claim_viols)
                print(f"\n  Customer Explanation:")
                print(textwrap.fill(result["explanation"], width=72, initial_indent="    ", subsequent_indent="    "))
                print(f"\n  Internal Summary:")
                print(f"    {result['violation_summary']}")
                
            print("\n" + "-" * 70)
        return
    except Exception as e:
        print(f"Bedrock call failed: {e}")
        print("Falling back to raw SHACL violations.\n")
    
    # Fallback: show raw violations
    print("Bedrock unavailable — showing raw SHACL violations instead.")
    print("In production, these violations are sent to Bedrock for explanation.\n")
    for claim in denied_claims:
        claim_uri = claim["claim_uri"]
        claim_viols = all_violations.get(claim_uri, [])
        print(f"Claim {claim['claim_id']} (${claim['claim_amount']:,.2f}) — Policy {claim['policy_id']}")
        for v in claim_viols:
            print(f"  Violation: {v['message']}")
        print()

explain_with_fallback(denied_claims, all_violations, data_graph)

---
### Module 9.5: Bedrock Guardrails - PII Detection on Denial Explanations

The denial explanations above are generated by an LLM. Before sending them to a patient, we should ensure they don’t accidentally leak PII (names, SSNs, addresses) that was present in the claim context.

**Bedrock Guardrails** provide pre-built PII detection and content safety filters that attach directly to `invoke_model` calls - no custom policy authoring required.

```
Claim Context + SHACL Violations
         │
         ▼
  ┌──────────────────┐
  │   Bedrock LLM    │
  │  (explain denial)│
  └────────┬─────────┘
           │
           ▼
  ┌──────────────────┐
  │   Guardrail      │
  │  PII → ANONYMIZE │
  │  SSN → BLOCK     │
  └────────┬─────────┘
           │
           ▼
  Safe explanation
  (PII redacted)
```

### Create the Guardrail

First, we create a Bedrock Guardrail configured to anonymize PII (names, addresses, phone numbers) and block sensitive identifiers (SSNs, credit card numbers). The guardrail persists in your account - this cell is idempotent:


In [ ]:
# ── 9.5.1: Create Bedrock Guardrail with PII Detection ───────────────
# This creates a guardrail that anonymizes PII and blocks SSNs.
# Run once — the guardrail persists in your account.
import boto3

GUARDRAIL_AVAILABLE = False
GUARDRAIL_ID = None
GUARDRAIL_VERSION = None

try:
    bedrock_mgmt = boto3.client("bedrock", region_name="us-east-1")

    # Check if guardrail already exists
    existing = bedrock_mgmt.list_guardrails(maxResults=100)
    for gr in existing.get("guardrails", []):
        if gr["name"] == "denial-explanation-pii-guard":
            GUARDRAIL_ID = gr["id"]
            GUARDRAIL_VERSION = gr["version"]
            print(f"Found existing guardrail: {GUARDRAIL_ID} (v{GUARDRAIL_VERSION})")
            break

    if not GUARDRAIL_ID:
        resp = bedrock_mgmt.create_guardrail(
            name="denial-explanation-pii-guard",
            description="PII detection and content safety for claim denial explanations",
            sensitiveInformationPolicyConfig={
                "piiEntitiesConfig": [
                    {"type": "EMAIL", "action": "ANONYMIZE"},
                    {"type": "PHONE", "action": "ANONYMIZE"},
                    {"type": "US_SOCIAL_SECURITY_NUMBER", "action": "BLOCK"},
                    {"type": "NAME", "action": "ANONYMIZE"},
                    {"type": "ADDRESS", "action": "ANONYMIZE"},
                    {"type": "CREDIT_DEBIT_CARD_NUMBER", "action": "BLOCK"},
                ]
            },
            contentPolicyConfig={
                "filtersConfig": [
                    {"type": "VIOLENCE", "inputStrength": "HIGH", "outputStrength": "HIGH"},
                    {"type": "INSULTS", "inputStrength": "HIGH", "outputStrength": "HIGH"},
                    {"type": "PROMPT_ATTACK", "inputStrength": "HIGH", "outputStrength": "NONE"},
                ]
            },
            blockedInputMessaging="Input blocked by guardrail.",
            blockedOutputsMessaging="Output blocked by guardrail due to policy violation.",
        )
        GUARDRAIL_ID = resp["guardrailId"]
        GUARDRAIL_VERSION = "DRAFT"
        print(f"Created guardrail: {GUARDRAIL_ID} (DRAFT)")

    GUARDRAIL_AVAILABLE = True

except Exception as e:
    print(f"Guardrail setup failed: {e}")
    print("Module 9.5 will skip guardrail-protected calls.")

### Re-run Denial Explanations with Guardrail

Now we re-run the same denial explanation pipeline from above, but with the guardrail attached to the `invoke_model` call. Compare the output - customer names and other PII should be anonymized in the explanations:


In [ ]:
import textwrap
import boto3
bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")
# ── 9.5.2: Re-run denial explanations WITH guardrail ─────────────────
# Same flow as Module 9, but invoke_model now includes the guardrail.
# Compare output — PII like customer names should be anonymized.

print(f"Guardrail: {GUARDRAIL_ID} (v{GUARDRAIL_VERSION})")
print(f"Model: us.anthropic.claude-haiku-4-5-20251001-v1:0\n")
print("=" * 70)

for claim in denied_claims:
    claim_uri = claim["claim_uri"]
    claim_viols = all_violations.get(claim_uri, [])
    if not claim_viols:
        continue

    claim_ctx = get_claim_context(data_graph, claim_uri)

    print(f"\nClaim {claim['claim_id']} (${claim['claim_amount']:,.2f})")
    print(f"Policy: {claim['policy_id']} | Incident: {claim['incident_date'][:10]}")

    # Build the same prompt the DenialExplainer uses
    explainer = DenialExplainer(model="haiku", region="us-east-1")
    prompt = explainer._build_prompt(claim_ctx, claim_viols)

    # Call Bedrock WITH guardrail attached
    response = bedrock.invoke_model(
        modelId="us.anthropic.claude-haiku-4-5-20251001-v1:0",
        guardrailIdentifier=GUARDRAIL_ID,
        guardrailVersion=GUARDRAIL_VERSION,
        body=json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 600,
            "temperature": 0.3,
            "messages": [{"role": "user", "content": prompt}],
        }),
    )

    result = json.loads(response["body"].read())
    content = result["content"][0]["text"]

    # Check guardrail action
    guardrail_action = result.get("amazon-bedrock-guardrailAction", "none")
    trace = result.get("amazon-bedrock-trace", {})

    print(f"  Guardrail action: {guardrail_action}")

    # Parse the explanation
    try:
        if "```json" in content:
            content = content.split("```json")[1].split("```")[0].strip()
        elif "```" in content:
            content = content.split("```")[1].split("```")[0].strip()
        parsed = json.loads(content)
        print(f"\n  Customer Explanation (guardrail-filtered):")
        print(textwrap.fill(parsed["explanation"], width=72, initial_indent="    ", subsequent_indent="    "))
    except Exception:
        print(f"\n  Raw output (guardrail-filtered):")
        print(textwrap.fill(content[:500], width=72, initial_indent="    ", subsequent_indent="    "))

    # Show if any PII was detected/redacted
    guardrail_output = trace.get("guardrail", {})
    if guardrail_output:
        assessments = guardrail_output.get("output", {}).get("outputAssessments", [{}])
        for assessment in assessments:
            pii_findings = assessment.get("sensitiveInformationPolicy", {}).get("piiEntities", [])
            if pii_findings:
                print(f"\n  PII detected and handled:")
                for pii in pii_findings:
                    print(f"    • {pii.get('type', '?')} → {pii.get('action', '?')}")

    print("\n" + "-" * 70)



→ Proceed to **07-Information-Extraction**
